<a href="https://colab.research.google.com/github/samyuktha07-prog/helphear-ai/blob/main/helphearAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# HELPHEAR AI — ADVANCED EMERGENCY COMMUNICATION ASSISTANT
# Google Colab | Python | Whisper ASR | NLP | Translation | TTS
# ============================================================

!pip -q install -U gradio transformers accelerate gTTS deep-translator

import os
import re
import tempfile
import torch
import gradio as gr
from transformers import pipeline
from deep_translator import GoogleTranslator
from gtts import gTTS

# -------------------- CONFIGURATION -------------------------

ASR_MODEL = "openai/whisper-small"
DEVICE = 0 if torch.cuda.is_available() else -1

print("Loading Whisper speech recognition model...")
asr = pipeline(
    "automatic-speech-recognition",
    model=ASR_MODEL,
    device=DEVICE
)
print("Whisper is ready!")

LANGUAGES = {
    "English": "en",
    "Tamil": "ta",
    "Hindi": "hi",
    "Telugu": "te"
}

# -------------------- INTENT DETECTION ---------------------

INTENTS = {
    "Medical assistance": [
        "medical", "doctor", "ambulance", "hospital", "injured",
        "bleeding", "pain", "sick", "unconscious", "medicine",
        "மருத்துவம்", "மருத்துவர்", "ஆம்புலன்ஸ்", "உடல்நிலை"
    ],
    "Fire emergency": [
        "fire", "smoke", "burning", "flames",
        "தீ", "புகை"
    ],
    "Accident": [
        "accident", "crash", "collision", "road accident",
        "விபத்து"
    ],
    "Personal safety": [
        "unsafe", "threat", "help me", "being followed",
        "danger", "trapped", "பாதுகாப்பு", "உதவி"
    ],
    "General assistance": [
        "help", "assistance", "support", "emergency",
        "உதவி வேண்டும்"
    ]
}

def detect_intent(text):
    text_lower = text.lower()
    scores = {}

    for category, keywords in INTENTS.items():
        scores[category] = sum(
            1 for word in keywords if word in text_lower
        )

    best = max(scores, key=scores.get)

    if scores[best] == 0:
        return "General assistance", "Low confidence"

    # Keyword matching is only a basic prototype classifier.
    confidence = "Keyword match — verify manually"
    return best, confidence


def clean_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    if text and text[-1] not in ".!?":
        text += "."
    return text


# -------------------- SPEECH RECOGNITION -------------------

def transcribe_audio(audio_path):
    if not audio_path:
        return ""

    result = asr(
        audio_path,
        generate_kwargs={"task": "transcribe"}
    )
    return clean_text(result.get("text", ""))


# -------------------- TRANSLATION --------------------------

def translate_text(text, target_language):
    target_code = LANGUAGES[target_language]

    try:
        return GoogleTranslator(
            source="auto",
            target=target_code
        ).translate(text)
    except Exception as e:
        return (
            "Translation unavailable. Check your internet connection. "
            f"Details: {str(e)[:180]}"
        )


# -------------------- MESSAGE GENERATION -------------------

def create_emergency_message(text, location, contact_name):
    intent, confidence = detect_intent(text)

    location = location.strip() or "Location not provided"
    contact_name = contact_name.strip() or "Emergency contact"

    message = (
        f"URGENT ASSISTANCE REQUEST\n"
        f"Category: {intent}\n"
        f"Message: {text}\n"
        f"Location: {location}\n"
        f"Intended recipient: {contact_name}\n"
        f"Please verify this request and arrange appropriate help."
    )

    return message, intent, confidence


# -------------------- TEXT-TO-SPEECH -----------------------

def make_speech(text, language):
    if not text.strip():
        return None

    # Keep speech output manageable.
    spoken_text = text[:2500]
    path = tempfile.mktemp(suffix=".mp3")

    try:
        gTTS(
            text=spoken_text,
            lang=LANGUAGES[language]
        ).save(path)
        return path
    except Exception as e:
        print("Speech generation error:", e)
        return None


# -------------------- MAIN PROCESSOR -----------------------

def process_request(
    audio,
    typed_message,
    location,
    contact_name,
    target_language
):
    # Use typed input if supplied; otherwise transcribe audio.
    transcript = (typed_message or "").strip()

    if not transcript and audio:
        try:
            transcript = transcribe_audio(audio)
        except Exception as e:
            return (
                "Transcription failed: " + str(e),
                "Unable to classify",
                "Please check the audio and try again.",
                None,
                "Check the transcript manually before using it."
            )

    if not transcript:
        return (
            "Please record audio or enter a message.",
            "Not detected",
            "",
            None,
            "No message was provided."
        )

    message, intent, confidence = create_emergency_message(
        transcript, location, contact_name
    )

    translated = translate_text(message, target_language)

    # Use a spoken-language message rather than the whole report.
    speech_text = translate_text(transcript, target_language)
    audio_output = make_speech(speech_text, target_language)

    report = (
        f"Detected category: {intent}\n"
        f"Classification note: {confidence}\n\n"
        f"Original transcript:\n{transcript}\n\n"
        f"Generated emergency message:\n{message}\n\n"
        f"Translated message ({target_language}):\n{translated}"
    )

    safety_note = (
        "REVIEW BEFORE USE. This prototype does not contact anyone, "
        "share your location, or dispatch emergency services. "
        "Confirm the transcript, translation, category, and location "
        "before sharing. For a real emergency in India, call 112 "
        "directly if you can do so safely."
    )

    return transcript, intent, report, audio_output, safety_note


# -------------------- GRADIO APPLICATION -------------------

with gr.Blocks(theme=gr.themes.Soft(), title="HelpHear AI") as app:
    gr.Markdown("""
    # 🆘 HelpHear AI
    ### Voice-Based Emergency Communication Assistant

    Speak or type a request. HelpHear creates a structured message,
    identifies a possible assistance category, translates it, and
    reads the translated request aloud.

    **Prototype only:** results must be checked by a person.
    """)

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎙️ Record or upload speech"
            )

            text_input = gr.Textbox(
                label="Or type your message",
                placeholder="Example: I need medical help near the bus stand...",
                lines=3
            )

            location_input = gr.Textbox(
                label="Location (optional)",
                placeholder="Enter a location manually"
            )

            contact_input = gr.Textbox(
                label="Intended recipient (optional)",
                placeholder="Example: Parent or trusted contact"
            )

            language_input = gr.Dropdown(
                choices=list(LANGUAGES.keys()),
                value="Tamil",
                label="Translate and speak in"
            )

            run_button = gr.Button(
                "Analyze and Generate Message",
                variant="primary"
            )

        with gr.Column():
            transcript_output = gr.Textbox(
                label="Recognized transcript",
                lines=3
            )

            intent_output = gr.Textbox(
                label="Possible assistance category"
            )

            report_output = gr.Textbox(
                label="Emergency report and translation",
                lines=14
            )

            speech_output = gr.Audio(
                label="🔊 Spoken message",
                type="filepath"
            )

    safety_output = gr.Textbox(
        label="Safety notice",
        lines=3
    )

    run_button.click(
        fn=process_request,
        inputs=[
            audio_input,
            text_input,
            location_input,
            contact_input,
            language_input
        ],
        outputs=[
            transcript_output,
            intent_output,
            report_output,
            speech_output,
            safety_output
        ]
    )

    gr.Markdown("""
    ---
    **Limitations:** Keyword-based categories may be wrong.
    Whisper and translation can make mistakes, particularly with
    mixed-language speech, background noise, names, and addresses.
    No message is automatically sent to an emergency contact.
    """)

app.launch(share=True, debug=False)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.7 MB/s eta 0:00:00
Loading Whisper speech recognition model...


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Whisper is ready!


/tmp/ipykernel_3057/1701073462.py:224: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="HelpHear AI") as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0ae19f58dfbae61d75.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
